# tpcve — оценка биомассы пшеницы (Kaggle)

Запуск трёх методов (`voxel`, `chm`, `alpha`) по dataset-спискам, вместе и раздельно по стадиям роста `--stage Z31` (дата 0828) и `--stage Z65` (дата 1002).

Каждый запуск: batch (признак по облакам) → автоматический analyze (регрессия `biomass ~ признак`, train+test).

**Требования:** в настройках ноутбука включить **Internet ON** (нужно для `git clone`) и подключить датасет `kobanur/wheat-biomass-and-lidar-dataset-yanco-2019`.

Результаты пишутся в `/kaggle/working/tpcve/results/` (volume CSV, regression CSV, графики).

## 1. Клонирование кода

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/Konabur/tpcve"
REPO_DIR = "/kaggle/working/tpcve"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    print("repo already cloned")

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

## 2. Установка зависимостей

`pip install -e .` ставит пакет editable и тянет зависимости из `pyproject.toml` (open3d, alphashape, scikit-learn, python-dotenv и т.д.). Импорты `import tpcve` / `import tools` работают из любого cwd.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "pip", "install", "--system", "-q", "-e", "."], check=True)
print("deps installed")

## 3. Пути к данным + проверка

Списки лежат в `DATA`, пути внутри списков относительны `DATA` → `--base-dir DATA`. Сначала убеждаемся, что первый файл из train-списка реально находится.

In [ ]:
from pathlib import Path
from tpcve.core.io import parse_list_line

DATA = Path("/kaggle/input/datasets/kobanur/wheat-biomass-and-lidar-dataset-yanco-2019/data/Yanco_TC_2019_HI-pcd")
TRAIN_LIST = DATA / ".." / "train_list.txt"
TEST_LIST  = DATA / ".." / "test_list.txt"

for p in (TRAIN_LIST, TEST_LIST):
    assert p.exists(), f"нет файла: {p}"

# sanity-check: первая валидная строка train-списка -> существующий .pcd
with open(TRAIN_LIST, encoding="utf-8") as f:
    for line in f:
        if line.strip() and not line.lstrip().startswith("#"):
            rel, labels = parse_list_line(line)
            full = DATA / rel.lstrip("/\\")
            print("rel  :", rel)
            print("full :", full)
            print("exists:", full.exists(), "| biomass:", labels["biomass"])
            assert full.exists(), "base-dir не совпадает с путями из списка — поправь DATA"
            break

In [ ]:
def run_batch(method, stage=None, *method_flags, **kwargs):
    """Запустить batch.py для одного метода/стадии (train+test, analyze в цепочке)."""
    cmd = [
        sys.executable, "batch.py",
        "--method", method,
        "--list", str(TRAIN_LIST),
        "--list-test", str(TEST_LIST),
        "--base-dir", str(DATA),
        *method_flags,
    ]
    if stage is not None:
        cmd.extend(["--stage", stage])
    print("\n$ " + " ".join(cmd) + "\n")
    p = subprocess.run(cmd, cwd=REPO_DIR, **kwargs)
    if p.returncode:
        raise SystemExit(f"batch завершился с кодом {p.returncode}")

## 4. Воксельный метод

`s (--voxel-sizes): 10, 15, 20, 25, 30, 35, 40, 45` (мм)

In [ ]:
run_batch("voxel", "--voxel-sizes", "10,15,20,25,30,35,40,45")

In [ ]:
run_batch("voxel", "Z31", "--voxel-sizes", "10,15,20,25,30,35,40,45")

In [ ]:
run_batch("voxel", "Z65", "--voxel-sizes", "10,15,20,25,30,35,40,45")

## 5. CHM (объём по сетке высот)

`c (--cell-sizes): 10, 15, 20, 25` (мм) · `p (--percentiles): 1, 5, 10, 50, 75, 95, 99`

In [ ]:
run_batch("chm", "--cell-sizes", "10,15,20,25", "--percentiles", "1,5,10,50,75,95,99")

In [ ]:
run_batch("chm", "Z31", "--cell-sizes", "10,15,20,25", "--percentiles", "1,5,10,50,75,95,99")

In [ ]:
run_batch("chm", "Z65", "--cell-sizes", "10,15,20,25", "--percentiles", "1,5,10,50,75,95,99")

## 6. Послойная альфа-форма

`v (--voxel-sizes): 10, 20, 30, 40, 50` (мм) · `α (--alphas): 20, 30, 40` · `dz (--layer-dz): 10, 20, 30, 40, 50, 60` (мм)

In [ ]:
env = {**os.environ, "TQDM_MININTERVAL": "30"}

In [ ]:
run_batch("alpha",
          "--voxel-sizes", "10,20,30,40,50",
          "--alphas", "20,30,40",
          "--layer-dz", "10,20,30,40,50,60",
          env=env)

In [ ]:
run_batch("alpha", "Z31",
          "--voxel-sizes", "10,20,30,40,50",
          "--alphas", "20,30,40",
          "--layer-dz", "10,20,30,40,50,60",
          env=env)

In [ ]:
run_batch("alpha", "Z65",
          "--voxel-sizes", "10,20,30,40,50",
          "--alphas", "20,30,40",
          "--layer-dz", "10,20,30,40,50,60",
          env=env)

## 7. Бейзлайны

`count` — число точек растительности → биомасса (простейший baseline). `percentile` — один скаляр (перцентиль высоты) → биомасса.

In [ ]:
run_batch("count")
run_batch("count", "Z31")
run_batch("count", "Z65")

In [ ]:
run_batch("percentile", "--percentiles", "1,5,10,50,75,95,99")
run_batch("percentile", "Z31", "--percentiles", "1,5,10,50,75,95,99")
run_batch("percentile", "Z65", "--percentiles", "1,5,10,50,75,95,99")

## 8. Результаты

Все CSV и графики — в `results/`. Ниже список свежих regression-CSV (отсортированы по R²) и упаковка в один архив для скачивания.

In [ ]:
import shutil

results_dir = Path(REPO_DIR) / "results"
print("regression CSV:")
for p in sorted(results_dir.glob("regression_csv/**/*.csv")):
    print("  ", p.relative_to(results_dir))

archive = shutil.make_archive("/kaggle/working/tpcve_results", "zip", results_dir)
print("\narchive:", archive)